Arquivo de teste 

In [0]:
# %python
import io, sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from src.config.settings import AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_KEY, BRONZE_CONTAINER
from azure.storage.blob import BlobServiceClient
import pandas as pd

conn_str = (f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};"
            f"AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net")
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
container_client = blob_service_client.get_container_client(BRONZE_CONTAINER)

arquivos = [b.name for b in container_client.list_blobs() if "/" not in b.name]

for nome in sorted(arquivos):
    blob_client = container_client.get_blob_client(nome)
    data = blob_client.download_blob().readall()
    pdf = pd.read_parquet(io.BytesIO(data))
    print(f"\n=== {nome} ({len(pdf)} linhas) ===")
    for col in pdf.columns:
        print(f"  {col}: {pdf[col].dtype}")

In [0]:
# %python
import io, sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from src.config.settings import AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_KEY, BRONZE_CONTAINER
from azure.storage.blob import BlobServiceClient
import pandas as pd

conn_str = (f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};"
            f"AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net")
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
container_client = blob_service_client.get_container_client(BRONZE_CONTAINER)

def ler(nome):
    data = container_client.get_blob_client(nome).download_blob().readall()
    return pd.read_parquet(io.BytesIO(data))

# Bases menores — verificação completa
for nome in ["2026-08-30_municipio.parquet",
             "2026-08-30_meta_alfabetizacao_municipio.parquet",
             "2026-08-30_uf.parquet",
             "2026-08-30_meta_alfabetizacao_uf.parquet",
             "2026-08-30_meta_alfabetizacao_brasil.parquet",
             "2026-08-30_ibge_municipios.parquet",
             "2026-08-30_ibge_estados.parquet"]:
    df = ler(nome)
    print(f"\n=== {nome} ({len(df)} linhas) ===")
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    print("Nulos:", nulos.to_dict() if len(nulos) else "nenhum")
    if "id_municipio" in df.columns:
        print("Tam id_municipio:", df["id_municipio"].astype(str).str.len().value_counts().to_dict())
        print("Duplicados id_municipio:", df["id_municipio"].duplicated().sum())

# Alunos — arquivo grande
df = ler("2026-08-30_alunos.parquet")
print(f"\n=== alunos.parquet ({len(df)} linhas) ===")
nulos = df.isna().sum()
nulos = nulos[nulos > 0]
print("Nulos:", nulos.to_dict() if len(nulos) else "nenhum")
print("Tam id_municipio:", df["id_municipio"].astype(str).str.len().value_counts().to_dict())
print("id_aluno duplicados:", df["id_aluno"].duplicated().sum())
print("Rede:", df["rede"].value_counts(dropna=False).to_dict())
print("Alfabetizado:", df["alfabetizado"].value_counts(dropna=False).to_dict())

In [0]:
# %python
import io, sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from src.config.settings import AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_KEY, BRONZE_CONTAINER
from azure.storage.blob import BlobServiceClient
import pandas as pd

conn_str = (f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};"
            f"AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net")
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
container_client = blob_service_client.get_container_client(BRONZE_CONTAINER)

def ler(nome):
    data = container_client.get_blob_client(nome).download_blob().readall()
    return pd.read_parquet(io.BytesIO(data))

def checar(df, nome, chaves):
    dups = df.duplicated(subset=chaves).sum()
    print(f"{nome}: {len(df)} linhas | chave {chaves} | duplicados: {dups}")

# Chaves compostas do desenho
checar(ler("2026-08-30_municipio.parquet"), "municipio",
       ["ano", "id_municipio", "serie", "rede"])
checar(ler("2026-08-30_uf.parquet"), "uf",
       ["ano", "sigla_uf", "serie", "rede"])
checar(ler("2026-08-30_meta_alfabetizacao_municipio.parquet"), "meta_municipio",
       ["ano", "id_municipio", "rede"])
checar(ler("2026-08-30_meta_alfabetizacao_uf.parquet"), "meta_uf",
       ["ano", "sigla_uf", "rede"])
checar(ler("2026-08-30_meta_alfabetizacao_brasil.parquet"), "meta_brasil",
       ["ano", "rede"])